# 20 Scikit-Learns' Tricks

In [1]:
import sklearn
import pandas as pd
import numpy as np

In [2]:
sklearn.__version__

'1.9.1'

## Trick 1: Random State

In [3]:
df = pd.read_csv('Datas/Titanic.csv')

In [4]:
x = df[['Age', 'Sex', 'Embarked']]
y = df.Survived

In [5]:
x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, random_state=1)
x_train.head(4)

,Age,Sex,Embarked
35,42.0,male,S
46,NaN,male,Q
453,49.0,male,C
291,19.0,female,C


In [6]:
x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, random_state=1)
x_train.head(4)

,Age,Sex,Embarked
35,42.0,male,S
46,NaN,male,Q
453,49.0,male,C
291,19.0,female,C


In [7]:
x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, random_state=3^1000)
x_train.head(4)

,Age,Sex,Embarked
614,35.00,male,S
803,0.42,male,C
771,48.00,male,S
418,30.00,male,S


## Trick 2: Stratify

In [8]:
df = pd.DataFrame({'x' : list(range(100)), 'y' : list(['not fraud']*92 + ['fraud']*8)})
df

,x,y
0,0,not fraud
1,1,not fraud
2,2,not fraud
3,3,not fraud
4,4,not fraud
...,...,...
95,95,fraud
96,96,fraud
97,97,fraud
98,98,fraud


In [9]:
df.tail(10)

,x,y
90,90,not fraud
91,91,not fraud
92,92,fraud
93,93,fraud
94,94,fraud
95,95,fraud
96,96,fraud
97,97,fraud
98,98,fraud
99,99,fraud


In [10]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(df.x, df.y, test_size=0.5, random_state=123)

In [11]:
print(f'train: {y_train.value_counts()}')
print(f'test: {y_test.value_counts()}')

train: y
not fraud    44
fraud         6
Name: count, dtype: int64
test: y
not fraud    48
fraud         2
Name: count, dtype: int64


In [12]:
# with stratify

In [13]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(df.x, df.y, test_size=0.5, stratify=df.y, random_state=123)

In [14]:
print(f'train: {y_train.value_counts()}')
print(f'test: {y_test.value_counts()}')

train: y
not fraud    46
fraud         4
Name: count, dtype: int64
test: y
not fraud    46
fraud         4
Name: count, dtype: int64


In [15]:
# the values are not devided equally. they are devided نسبت به درصد تقسیم ترین و تست.

## Trick 3: Missing Indicator

In [16]:
df = pd.DataFrame({'Age' : [10, 20 , np.nan, 40, np.nan]})
df

,Age
0,10.0
1,20.0
2,NaN
3,40.0
4,NaN


In [17]:
imputer = sklearn.impute.SimpleImputer()

In [18]:
imputer.fit_transform(df)

array([[10.        ],
       [20.        ],
       [23.33333333],
       [40.        ],
       [23.33333333]])

In [19]:
# اندیکاتور و تمام تنظیماتی که در داده تغییر ایجاد میکنن باید توی مرحله اول ذخیره سازی کلاس انجام بشه. چون قرار هست فیت بشه.

In [20]:
imputer2 = sklearn.impute.SimpleImputer(add_indicator=True)

In [21]:
imputer2.fit_transform(df)

array([[10.        ,  0.        ],
       [20.        ,  0.        ],
       [23.33333333,  1.        ],
       [40.        ,  0.        ],
       [23.33333333,  1.        ]])

In [22]:
df2 = pd.concat([df, df], axis=1)
df2

,Age,Age
0,10.0,10.0
1,20.0,20.0
2,NaN,NaN
3,40.0,40.0
4,NaN,NaN


In [23]:
df2.columns = ['Age1', 'Age2']

In [24]:
imputer2.fit_transform(df2)

array([[10.        , 10.        ,  0.        ,  0.        ],
       [20.        , 20.        ,  0.        ,  0.        ],
       [23.33333333, 23.33333333,  1.        ,  1.        ],
       [40.        , 40.        ,  0.        ,  0.        ],
       [23.33333333, 23.33333333,  1.        ,  1.        ]])

## Trick 4 & 5: Pipeline & Columntransformer

In [25]:
knc = sklearn.neighbors.KNeighborsClassifier()
scaler = sklearn.preprocessing.StandardScaler()
num_imputer = sklearn.impute.SimpleImputer(strategy='mean')
cat_imputer = sklearn.impute.SimpleImputer(strategy='most_frequent')
ohe = sklearn.preprocessing.OneHotEncoder(drop= 'first')

In [26]:
df = pd.read_csv('Datas/Titanic.csv')

In [27]:
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [28]:
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

In [29]:
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [30]:
df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [31]:
num_pipe = sklearn.pipeline.Pipeline([
    ('imputer', num_imputer),
    ('scaler', scaler)
])
cat_pipe = sklearn.pipeline.Pipeline([
    ('imputer', cat_imputer),
    ('ohe', ohe)
])

In [32]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer([
    ('numerical', num_pipe, ['Age', 'Fare']),
    ('categorical', cat_pipe, ['Pclass', 'Sex', 'Embarked'])
], remainder='passthrough')

In [33]:
pipeline = sklearn.pipeline.Pipeline([
                                     ('transformer', transformer),
                                     ('model', knc)
                                     ])

In [34]:
x = df.drop('Survived', axis=1)
y = df.Survived

In [35]:
x_train, x_test, y_train, y_test = sklearn.model_selection.train_test_split(x, y, test_size=0.3, random_state=0)

In [36]:
pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('transformer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Pclass','Sex','Age',...,'Parch','Fare','Embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remainin

In [38]:
pipeline.score(x_train, y_train)

0.85553772070626

In [37]:
pipeline.score(x_test, y_test)

0.7798507462686567

## Trick 6: Selecting columns for ColumnTransformer

In [39]:
#there are some ways

In [40]:
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector

In [41]:
from sklearn.preprocessing import OneHotEncoder

In [42]:
df = pd.read_csv('Datas/Titanic.csv')
cols = ['Age', 'Embarked', 'Sex', 'Fare']
df = df[cols]
df.head(5)

,Age,Embarked,Sex,Fare
0,22.0,S,male,7.2500
1,38.0,C,female,71.2833
2,26.0,S,female,7.9250
3,35.0,S,female,53.1000
4,35.0,S,male,8.0500


In [43]:
ohe = OneHotEncoder()

In [44]:
ct = make_column_transformer((ohe, ['Embarked', 'Sex']))
ct = make_column_transformer((ohe, [1, 2]))
ct = make_column_transformer((ohe, slice(1, 3))) # wrong : [1:3] . that works just after an iterable
ct = make_column_transformer((ohe, make_column_selector(pattern='E|S')))
ct = make_column_transformer((ohe, make_column_selector(dtype_include='object')))
ct = make_column_transformer((ohe, make_column_selector(dtype_exclude='number')))

## Trick 7: 

## Trick 8: 

## Trick 9: 

## Trick 10: 

## Trick 11: 

## Trick 12: 

## Trick 13: 

## Trick 14: 

## Trick 15: 